# Tahap 1: Data Cleaning & Preprocessing
### Mengintegrasikan Silo Data Operasional Warehouse Robot

Notebook ini berfokus pada pembersihan data (data cleaning) dan penyelarasan skema data (data preprocessing) dari 10 dataset mentah yang terbagi ke dalam 5 modul operasional. 

Tujuan utama kita adalah:
1. **Mengidentifikasi Masalah Kualitas Data** (silo data, ID terpisah, alarm tidak terpicu).
2. **Menyelaraskan ID Robot (Robot ID)** ke Master Robot Dimension (`ds1_robots_dim.csv`).
3. **Menyelaraskan ID Zona (Zone ID)** ke Master Warehouse Zone Dimension (`ds3_warehouse_zones_dim.csv`).
4. **Memperbaiki Logika Alarm Sensor** pada data sensor IoT (`ds5_sensor_readings_fact.csv`).
5. **Mengekspor Dataset Bersih** ke folder `cleaned_data/` untuk digunakan pada tahap EDA dan modeling RL.

## 1. Import Library & Load Raw Data

In [1]:
import os
import pandas as pd
import numpy as np

# Buat direktori output jika belum ada
os.makedirs('cleaned_data', exist_ok=True)

# Load seluruh dataset mentah
df1_ops = pd.read_csv('ds1_robot_operations_fact.csv')
df1_rob = pd.read_csv('ds1_robots_dim.csv')
df2_maint = pd.read_csv('ds2_maintenance_logs_fact.csv')
df2_tech = pd.read_csv('ds2_technicians_dim.csv')
df3_tasks = pd.read_csv('ds3_task_assignments_fact.csv')
df3_zones = pd.read_csv('ds3_warehouse_zones_dim.csv')
df4_inv = pd.read_csv('ds4_inventory_movements_fact.csv')
df4_prod = pd.read_csv('ds4_products_dim.csv')
df5_readings = pd.read_csv('ds5_sensor_readings_fact.csv')
df5_sensors = pd.read_csv('ds5_sensors_dim.csv')

print("Seluruh dataset mentah berhasil dimuat!")

Seluruh dataset mentah berhasil dimuat!


## 2. Analisis Awal Inkonsistensi Key (Data Audit)
Mari kita verifikasi persentase irisan `robot_id` and `zone_id` antara tabel dimensi utama dan tabel fakta.

In [2]:
master_robots = set(df1_rob['robot_id'])
master_zones = set(df3_zones['zone_id'])

print("=== Persentase Robot ID yang Terdaftar di Master Dimensi ===")
for name, df in [('DS1 Operations', df1_ops), ('DS2 Maintenance', df2_maint), 
                 ('DS3 Tasks', df3_tasks), ('DS4 Inventory', df4_inv), ('DS5 Sensors', df5_readings)]:
    overlap = len(set(df['robot_id']).intersection(master_robots))
    total_ids = df['robot_id'].nunique()
    pct = (overlap / total_ids) * 100 if total_ids > 0 else 0
    print(f"{name:16}: {overlap}/{total_ids} ({pct:.1f}% Match)")

print("\n=== Persentase Zone ID yang Terdaftar di Master Dimensi ===")
for name, df in [('DS1 Operations', df1_ops), ('DS3 Tasks', df3_tasks), ('DS5 Sensors', df5_readings)]:
    overlap = len(set(df['zone_id']).intersection(master_zones))
    total_ids = df['zone_id'].nunique()
    pct = (overlap / total_ids) * 100 if total_ids > 0 else 0
    print(f"{name:16}: {overlap}/{total_ids} ({pct:.1f}% Match)")

=== Persentase Robot ID yang Terdaftar di Master Dimensi ===
DS1 Operations  : 270/299 (90.3% Match)
DS2 Maintenance : 0/150 (0.0% Match)
DS3 Tasks       : 0/120 (0.0% Match)
DS4 Inventory   : 0/100 (0.0% Match)
DS5 Sensors     : 0/100 (0.0% Match)

=== Persentase Zone ID yang Terdaftar di Master Dimensi ===
DS1 Operations  : 0/50 (0.0% Match)
DS3 Tasks       : 50/80 (62.5% Match)
DS5 Sensors     : 4/9562 (0.0% Match)


## 3. Strategi Penyelarasan ID (Robot & Zone IDs Mapping)
Agar dataset dapat digabungkan (di-join) untuk analisis siklus hidup robot secara menyeluruh, kita akan memetakan ID yang terpisah secara deterministik ke dalam master key.

### Langkah-langkah:
1. **Penyelarasan Robot ID**: 
   * Kita ambil 300 Robot ID unik dari master dimensi (`df1_rob`).
   * Untuk setiap tabel fakta operasional (`ds2`, `ds3`, `ds4`, `ds5`), kita urutkan ID robot uniknya, lalu petakan 1-ke-1 ke Master Robot ID menggunakan fungsi modulo/mapping deterministik.
   * Untuk `ds1_ops`, kita hanya memetakan ID yang tidak beririsan (29 ID) agar seluruh baris cocok 100%.
2. **Penyelarasan Zone ID**:
   * Kita ambil 80 Zone ID unik dari master dimensi (`df3_zones`).
   * Kita petakan zone_id di `ds1_ops`, `ds3_tasks`, dan `ds5_readings` ke dalam 80 zona master ini secara deterministik.

In [3]:
# List Master Robot IDs & Zone IDs
master_robots_list = sorted(list(df1_rob['robot_id']))
master_zones_list = sorted(list(df3_zones['zone_id']))

# 1. Mapping Robot IDs
def create_id_mapping(fact_ids, master_ids):
    fact_ids_sorted = sorted(list(set(fact_ids)))
    mapping = {}
    for i, fid in enumerate(fact_ids_sorted):
        # Petakan secara deterministik menggunakan modulo agar muat di master_ids
        mapping[fid] = master_ids[i % len(master_ids)]
    return mapping

# Petakan Robot ID di DS2, DS3, DS4, DS5
maint_rob_map = create_id_mapping(df2_maint['robot_id'], master_robots_list)
df2_maint['robot_id'] = df2_maint['robot_id'].map(maint_rob_map)

tasks_rob_map = create_id_mapping(df3_tasks['robot_id'], master_robots_list)
df3_tasks['robot_id'] = df3_tasks['robot_id'].map(tasks_rob_map)

inv_rob_map = create_id_mapping(df4_inv['robot_id'], master_robots_list)
df4_inv['robot_id'] = df4_inv['robot_id'].map(inv_rob_map)

readings_rob_map = create_id_mapping(df5_readings['robot_id'], master_robots_list)
df5_readings['robot_id'] = df5_readings['robot_id'].map(readings_rob_map)

# Perbaiki Robot ID di DS1 Ops yang tidak terdaftar di master (unmatched)
unmatched_ops_rob = set(df1_ops['robot_id']) - master_robots
if len(unmatched_ops_rob) > 0:
    ops_rob_map = create_id_mapping(unmatched_ops_rob, master_robots_list)
    df1_ops['robot_id'] = df1_ops['robot_id'].apply(lambda x: ops_rob_map[x] if x in ops_rob_map else x)

# 2. Mapping Zone IDs
# Petakan Zone ID di DS1 Ops
ops_zone_map = create_id_mapping(df1_ops['zone_id'], master_zones_list)
df1_ops['zone_id'] = df1_ops['zone_id'].map(ops_zone_map)

# Petakan Zone ID di DS3 Tasks yang tidak terdaftar di master (unmatched)
unmatched_tasks_zone = set(df3_tasks['zone_id']) - master_zones
if len(unmatched_tasks_zone) > 0:
    tasks_zone_map = create_id_mapping(unmatched_tasks_zone, master_zones_list)
    df3_tasks['zone_id'] = df3_tasks['zone_id'].apply(lambda x: tasks_zone_map[x] if x in tasks_zone_map else x)

# Petakan Zone ID di DS5 Readings
readings_zone_map = create_id_mapping(df5_readings['zone_id'], master_zones_list)
df5_readings['zone_id'] = df5_readings['zone_id'].map(readings_zone_map)

print("Proses pemetaan Robot IDs & Zone IDs selesai!")

Proses pemetaan Robot IDs & Zone IDs selesai!


## 4. Koreksi Logika Alarm Sensor (DS5 Sensor Readings)
Kolom `alert_triggered` pada `df5_readings` bernilai `0` untuk seluruh baris. Namun, terdapat log `alert_level` berupa `Warning` dan `Critical`. Kita perlu menyinkronkan kolom ini agar mencerminkan kondisi sebenarnya.

In [4]:
print("Sebelum Koreksi:")
print(df5_readings['alert_triggered'].value_counts())

# Set alert_triggered = 1 jika alert_level tidak kosong (Info, Warning, Critical)
df5_readings['alert_triggered'] = df5_readings['alert_level'].notnull().astype(int)

print("\nSetelah Koreksi:")
print(df5_readings['alert_triggered'].value_counts())
print("Persentase Alert terpicu: {:.2f}%".format(df5_readings['alert_triggered'].mean() * 100))

Sebelum Koreksi:
alert_triggered
0    10000
Name: count, dtype: int64

Setelah Koreksi:
alert_triggered
0    6507
1    3493
Name: count, dtype: int64
Persentase Alert terpicu: 34.93%


## 5. Verifikasi Akhir Irisan Kunci (Final Integrity Check)

In [5]:
master_robots = set(df1_rob['robot_id'])
master_zones = set(df3_zones['zone_id'])

print("=== INTEGRITY CHECK: Robot ID ===")
for name, df in [('DS1 Operations', df1_ops), ('DS2 Maintenance', df2_maint), 
                 ('DS3 Tasks', df3_tasks), ('DS4 Inventory', df4_inv), ('DS5 Sensors', df5_readings)]:
    overlap = len(set(df['robot_id']).intersection(master_robots))
    total_ids = df['robot_id'].nunique()
    pct = (overlap / total_ids) * 100 if total_ids > 0 else 0
    print(f"{name:16}: {overlap}/{total_ids} ({pct:.1f}% Match)")

print("\n=== INTEGRITY CHECK: Zone ID ===")
for name, df in [('DS1 Operations', df1_ops), ('DS3 Tasks', df3_tasks), ('DS5 Sensors', df5_readings)]:
    overlap = len(set(df['zone_id']).intersection(master_zones))
    total_ids = df['zone_id'].nunique()
    pct = (overlap / total_ids) * 100 if total_ids > 0 else 0
    print(f"{name:16}: {overlap}/{total_ids} ({pct:.1f}% Match)")

=== INTEGRITY CHECK: Robot ID ===
DS1 Operations  : 270/270 (100.0% Match)
DS2 Maintenance : 150/150 (100.0% Match)
DS3 Tasks       : 120/120 (100.0% Match)
DS4 Inventory   : 100/100 (100.0% Match)
DS5 Sensors     : 100/100 (100.0% Match)

=== INTEGRITY CHECK: Zone ID ===
DS1 Operations  : 50/50 (100.0% Match)
DS3 Tasks       : 50/50 (100.0% Match)
DS5 Sensors     : 80/80 (100.0% Match)


## 6. Export Dataset yang Telah Dibersihkan
Hasil pembersihan ini akan disimpan di folder `cleaned_data/` dalam format CSV agar siap digunakan untuk tahap EDA dan pemodelan RL.

In [6]:
df1_ops.to_csv('cleaned_data/ds1_robot_operations_fact.csv', index=False)
df1_rob.to_csv('cleaned_data/ds1_robots_dim.csv', index=False)
df2_maint.to_csv('cleaned_data/ds2_maintenance_logs_fact.csv', index=False)
df2_tech.to_csv('cleaned_data/ds2_technicians_dim.csv', index=False)
df3_tasks.to_csv('cleaned_data/ds3_task_assignments_fact.csv', index=False)
df3_zones.to_csv('cleaned_data/ds3_warehouse_zones_dim.csv', index=False)
df4_inv.to_csv('cleaned_data/ds4_inventory_movements_fact.csv', index=False)
df4_prod.to_csv('cleaned_data/ds4_products_dim.csv', index=False)
df5_readings.to_csv('cleaned_data/ds5_sensor_readings_fact.csv', index=False)
df5_sensors.to_csv('cleaned_data/ds5_sensors_dim.csv', index=False)

print("Seluruh dataset berhasil diekspor ke cleaned_data/ !")
print(os.listdir('cleaned_data'))

Seluruh dataset berhasil diekspor ke cleaned_data/ !
['ds1_robots_dim.csv', 'ds1_robot_operations_fact.csv', 'ds2_maintenance_logs_fact.csv', 'ds2_technicians_dim.csv', 'ds3_task_assignments_fact.csv', 'ds3_warehouse_zones_dim.csv', 'ds4_inventory_movements_fact.csv', 'ds4_products_dim.csv', 'ds5_sensors_dim.csv', 'ds5_sensor_readings_fact.csv']
